# CeNN running

In [ ]:
import os
os.chdir('/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16')

In [ ]:
#!/usr/bin/python
# -*- coding: utf-8 -*-

"""
The MIT License (MIT)

Copyright (c) 2014 Ankit Aggarwal <ankitaggarwal011@gmail.com>

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

from __future__ import print_function
import scipy.signal as sig
import scipy.integrate as sint
from PIL import Image as img
import numpy as np
import os.path
import warnings

SUPPORTED_FILETYPES = (
    'jpeg', 'jpg', 'png', 'tiff', 'gif', 'bmp',
)

warnings.filterwarnings('ignore')  # Ignore trivial warnings


class PyCNN(object):
    """Image Processing with Cellular Neural Networks (CNN).

    Cellular Neural Networks (CNN) are a parallel computing paradigm that was
    first proposed in 1988. Cellular neural networks are similar to neural
    networks, with the difference that communication is allowed only between
    neighboring units. Image Processing is one of its applications. CNN
    processors were designed to perform image processing; specifically, the
    original application of CNN processors was to perform real-time ultra-high
    frame-rate (>10,000 frame/s) processing unachievable by digital processors.

    This python library is the implementation of CNN for the application of
    Image Processing.


    Attributes:
        n (int): Height of the image.
        m (int): Width of the image.
    """

    def __init__(self):
        """Sets the initial class attributes m (width) and n (height)."""
        self.m = 0  # width (number of columns)
        self.n = 0  # height (number of rows)

    def f(self, t, x, Ib, Bu, tempA):
        """Computes the derivative of x at t.

        Args:
            x: The input.
            Ib (float): System bias.
            Bu: Convolution of control template with input.
            tempA (:obj:`list` of :obj:`list`of :obj:`float`): Feedback
                template.
        """
        x = x.reshape((self.n, self.m))
        dx = -x + Ib + Bu + sig.convolve2d(self.cnn(x), tempA, 'same')
        return dx.reshape(self.m * self.n)

    def cnn(self, x):
        """Piece-wise linear sigmoid function.

        Args:
            x : Input to the piece-wise linear sigmoid function.
        """
        return 0.5 * (abs(x + 1) - abs(x - 1))

    def validate(self, inputLocation):
        """Checks if a string path exists or is from a supported file type.

        Args:
            inputLocation (str): A string with the path to the image.

        Raises:
            IOError: If `inputLocation` does not exist or is not a file.
            Exception: If file type is not supported.
        """
        _, ext = os.path.splitext(inputLocation)
        ext = ext.lstrip('.').lower()
        if not os.path.exists(inputLocation):
            raise IOError('File {} does not exist.'.format(inputLocation))
        elif not os.path.isfile(inputLocation):
            raise IOError('Path {} is not a file.'.format(inputLocation))
        elif ext not in SUPPORTED_FILETYPES:
            raise Exception(
                '{} file type is not supported. Supported: {}'.format(
                    ext, ', '.join(SUPPORTED_FILETYPES)
                )
            )

    # tempA: feedback template, tempB: control template
    def imageProcessing(self, inputLocation, outputLocation,
                        tempA, tempB, initialCondition, Ib, t):
        """Process the image with the input arguments.

        Args:
            inputLocation (str): The string path for the input image.
            outputLocation (str): The string path for the output processed
                image.
            tempA (:obj:`list` of :obj:`list`of :obj:`float`): Feedback
                template.
            tempB (:obj:`list` of :obj:`list`of :obj:`float`): Control
                template.
            initialCondition (float): The initial state.
            Ib (float): System bias.
            t (numpy.ndarray): A numpy array with evenly spaced numbers
                representing time points.
        """
        gray = img.open(inputLocation).convert('RGB')
        self.m, self.n = gray.size
        u = np.array(gray)
        u = u[:, :, 0]
        z0 = u * initialCondition
        Bu = sig.convolve2d(u, tempB, 'same')
        z0 = z0.flatten()
        tFinal = t.max()
        tInitial = t.min()
        if t.size > 1:
            dt = t[1] - t[0]
        else:
            dt = t[0]
        ode = sint.ode(self.f) \
            .set_integrator('vode') \
            .set_initial_value(z0, tInitial) \
            .set_f_params(Ib, Bu, tempA)
        while ode.successful() and ode.t < tFinal + 0.1:
            ode_result = ode.integrate(ode.t + dt)
        z = self.cnn(ode_result)

        # out_l = z[:].reshape((self.n, self.m))
        # out_l = out_l / (255.0)
        # out_l = np.uint8(np.round(out_l * 255))

        # out_l = img.fromarray(out_l).convert('RGB')
        # out_l.save(outputLocation)

        out_l = z[:].reshape((self.n, self.m))      # (H, W)
        out_l = out_l.astype(np.float32) / 255.0    # normalize to [0, 1]

        # Save as .npy (make sure outputLocation ends with ".npy")
        np.save(outputLocation, out_l)

    # general image processing for given templates
    def generalTemplates(self,
                         name='Image processing',
                         inputLocation='',
                         outputLocation='output.png',
                         tempA_A=[[0.0, 0.0, 0.0],
                                  [0.0, 0.0, 0.0],
                                  [0.0, 0.0, 0.0]],
                         tempB_B=[[0.0, 0.0, 0.0],
                                  [0.0, 0.0, 0.0],
                                  [0.0, 0.0, 0.0]],
                         initialCondition=0.0,
                         Ib_b=0.0,
                         t=np.linspace(0, 10.0, num=2)):
        """Validate and process the image with the input arguments.

        Args:
            name (str): The name of the template.
            inputLocation (str): The string path for the input image.
            outputLocation (str): The string path for the output processed
                image.
            tempA_A (:obj:`list` of :obj:`list`of :obj:`float`): Feedback
                template.
            tempB_B (:obj:`list` of :obj:`list`of :obj:`float`): Control
                template.
            initialCondition (float): The initial state.
            Ib_b (float): System bias.
            t (numpy.ndarray): A numpy array with evenly spaced numbers
                representing time points.
        """
        self.validate(inputLocation)
        print(name, 'initialized.')
        self.imageProcessing(inputLocation,
                             outputLocation,
                             np.array(tempA_A),
                             np.array(tempB_B),
                             initialCondition,
                             Ib_b,
                             t)
        print('Processing on image %s is complete' % (inputLocation))
        print('Result is saved at %s.\n' % (outputLocation))

    def edgeDetection(self, inputLocation='', outputLocation='output.png'):
        """Performs Edge Detection on the input image.

        The output is a binary image showing all edges of the input image in
        black.

        A = [[0.0 0.0 0.0],
             [0.0 1.0 0.0],
             [0.0 0.0 0.0]]

        B = [[−1.0 −1.0 −1.0],
             [−1.0 8.0 −1.0],
             [−1.0 −1.0 −1.0]]

        z = −1.0

        Initial state = 0.0

        Args:
            inputLocation (str): The string path for the input image.
            outputLocation (str): The string path for the output processed
                image.
        """
        name = 'Edge detection'
        tempA = [[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]]
        tempB = [[-1.0, -1.0, -1.0], [-1.0, 8.0, -1.0], [-1.0, -1.0, -1.0]]
        Ib = -1.0
        # num refers to the number of samples of time points from start = 0 to
        # end = 10.0
        t = np.linspace(0, 10.0, num=2)
        # some image processing methods might require more time point samples.
        initialCondition = 0.0
        self.generalTemplates(
            name,
            inputLocation,
            outputLocation,
            tempA,
            tempB,
            initialCondition,
            Ib,
            t)

    def grayScaleEdgeDetection(self, inputLocation='',
                               outputLocation='output.png'):
        """Performs Gray-scale Edge Detection on the input image.

        The output is a Gray-scale image showing an edge map of the input
        image in black.

        A = [[0.0 0.0 0.0],
             [0.0 2.0 0.0],
             [0.0 0.0 0.0]]

        B = [[−1.0 −1.0 −1.0],
             [−1.0 8.0 −1.0],
             [−1.0 −1.0 −1.0]]

        z = −0.5

        Initial state = 0.0

        Args:
            inputLocation (str): The string path for the input image.
            outputLocation (str): The string path for the output processed
                image.
        """
        name = 'Grayscale edge detection'
        tempA = [[0.0, 0.0, 0.0], [0.0, 2.0, 0.0], [0.0, 0.0, 0.0]]
        tempB = [[-1.0, -1.0, -1.0], [-1.0, 8.0, -1.0], [-1.0, -1.0, -1.0]]
        Ib = -0.5
        t = np.linspace(0, 1.0, num=101)
        initialCondition = 0.0
        self.generalTemplates(
            name,
            inputLocation,
            outputLocation,
            tempA,
            tempB,
            initialCondition,
            Ib,
            t)

In [ ]:
# Initialize object
cnn = PyCNN()

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt


def prepare_cenn_images(root_dir: str, cnn) -> None:
    """
    root_dir: path to the root 'test' folder.
    """
    root = Path(root_dir)

    # 1. Create output folders
    cenn_root = root / "cenn_images"
    edge_dir = cenn_root / "edgeDetection"
    gray_dir = cenn_root / "grayScaleEdgeDetection"
    tmp_dir = root / "tmp"

    edge_dir.mkdir(parents=True, exist_ok=True)
    gray_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # 2. Locate .npy files inside images folder
    images_dir = root / "images"
    npy_files = sorted(images_dir.glob("*.npy"))

    if not npy_files:
        print(f"No .npy files found in {images_dir}")
        return

    # 3. Process each .npy file
    for npy_path in tqdm(npy_files):
        image_name = npy_path.stem  # filename without extension

        # Output paths for this image
        edge_output_npy = edge_dir / f"{image_name}.npy"
        gray_output_npy = gray_dir / f"{image_name}.npy"

        # ---- Skip if already processed ----
        if edge_output_npy.exists() and gray_output_npy.exists():
            print(f"Skipping {image_name}: outputs already exist.")
            continue

        # Load array
        image = np.load(npy_path)

        # Save temporary PNG in ./tmp
        tmp_png_path = tmp_dir / f"{image_name}.png"
        plt.imsave(tmp_png_path, image, cmap="gray")

        # Run only what is missing, in case of partial previous runs
        if not edge_output_npy.exists():
            cnn.edgeDetection(str(tmp_png_path), str(edge_output_npy))
            print(f"  edgeDetection done: {edge_output_npy}")
        else:
            print(f"  edgeDetection skipped (exists): {edge_output_npy}")

        if not gray_output_npy.exists():
            cnn.grayScaleEdgeDetection(str(tmp_png_path), str(gray_output_npy))
            print(f"  grayScaleEdgeDetection done: {gray_output_npy}")
        else:
            print(f"  grayScaleEdgeDetection skipped (exists): {gray_output_npy}")

        print(f"Finished {image_name}")



In [ ]:
prepare_cenn_images(
    root_dir="/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train",
    cnn=cnn,
)

 16%|█▋        | 179/1095 [00:04<00:14, 64.99it/s]

Skipping images_0: outputs already exist.
Skipping images_1: outputs already exist.
Skipping images_10: outputs already exist.
Skipping images_100: outputs already exist.
Skipping images_1000: outputs already exist.
Skipping images_1001: outputs already exist.
Skipping images_1002: outputs already exist.
Skipping images_1003: outputs already exist.
Skipping images_1004: outputs already exist.
Skipping images_1005: outputs already exist.
Skipping images_1006: outputs already exist.
Skipping images_1007: outputs already exist.
Skipping images_1008: outputs already exist.
Skipping images_1009: outputs already exist.
Skipping images_101: outputs already exist.
Skipping images_1010: outputs already exist.
Skipping images_1011: outputs already exist.
Skipping images_1012: outputs already exist.
Skipping images_1013: outputs already exist.
Skipping images_1014: outputs already exist.
Skipping images_1015: outputs already exist.
Skipping images_1016: outputs already exist.
Skipping images_1017

 33%|███▎      | 359/1095 [00:04<00:04, 174.97it/s]

Skipping images_174: outputs already exist.
Skipping images_175: outputs already exist.
Skipping images_176: outputs already exist.
Skipping images_177: outputs already exist.
Skipping images_178: outputs already exist.
Skipping images_179: outputs already exist.
Skipping images_18: outputs already exist.
Skipping images_180: outputs already exist.
Skipping images_181: outputs already exist.
Skipping images_182: outputs already exist.
Skipping images_183: outputs already exist.
Skipping images_184: outputs already exist.
Skipping images_185: outputs already exist.
Skipping images_186: outputs already exist.
Skipping images_187: outputs already exist.
Skipping images_188: outputs already exist.
Skipping images_189: outputs already exist.
Skipping images_19: outputs already exist.
Skipping images_190: outputs already exist.
Skipping images_191: outputs already exist.
Skipping images_192: outputs already exist.
Skipping images_193: outputs already exist.
Skipping images_194: outputs alrea

 53%|█████▎    | 583/1095 [00:04<00:01, 379.80it/s]

...

Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/tmp/images_644.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/grayScaleEdgeDetection/images_644.npy.

  grayScaleEdgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/grayScaleEdgeDetection/images_644.npy
Finished images_644
Edge detection initialized.
Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/tmp/images_645.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/edgeDetection/images_645.npy.

  edgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/edgeDetection/images_645.npy
Grayscale edge detection initialized.


 64%|██████▍   | 703/1095 [59:13<3:40:19, 33.72s/it]

Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/tmp/images_645.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/grayScaleEdgeDetection/images_645.npy.

  grayScaleEdgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/grayScaleEdgeDetection/images_645.npy
Finished images_645
Edge detection initialized.
Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/tmp/images_646.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/edgeDetection/images_646.npy.

  edgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/edgeDetection/images_646.npy
Grayscale edge detection initialized.


 64%|██████▍   | 704/1095 [59:48<3:41:54, 34.05s/it]

Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/tmp/images_646.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/grayScaleEdgeDetection/images_646.npy.

  grayScaleEdgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/train/cenn_images/grayScaleEdgeDetection/images_646.npy
Finished images_646
Edge detection initialized.


In [ ]:
prepare_cenn_images(
    root_dir="/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test",
    cnn=cnn,
)

 31%|███       | 110/357 [00:01<00:01, 134.78it/s]

Skipping images_0: outputs already exist.
Skipping images_1: outputs already exist.
Skipping images_10: outputs already exist.
Skipping images_100: outputs already exist.
Skipping images_101: outputs already exist.
Skipping images_102: outputs already exist.
Skipping images_103: outputs already exist.
Skipping images_104: outputs already exist.
Skipping images_105: outputs already exist.
Skipping images_106: outputs already exist.
Skipping images_107: outputs already exist.
Skipping images_108: outputs already exist.
Skipping images_109: outputs already exist.
Skipping images_11: outputs already exist.
Skipping images_110: outputs already exist.
Skipping images_111: outputs already exist.
Skipping images_112: outputs already exist.
Skipping images_113: outputs already exist.
Skipping images_114: outputs already exist.
Skipping images_115: outputs already exist.
Skipping images_116: outputs already exist.
Skipping images_117: outputs already exist.
Skipping images_118: outputs already e

 31%|███       | 110/357 [00:16<00:01, 134.78it/s]

Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/tmp/images_226.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/edgeDetection/images_226.npy.

  edgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/edgeDetection/images_226.npy
Grayscale edge detection initialized.


 40%|████      | 143/357 [00:34<01:05,  3.25it/s] 

...

Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/tmp/images_97.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/grayScaleEdgeDetection/images_97.npy.

  grayScaleEdgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/grayScaleEdgeDetection/images_97.npy
Finished images_97
Edge detection initialized.
Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/tmp/images_98.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/edgeDetection/images_98.npy.

  edgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/edgeDetection/images_98.npy
Grayscale edge detection initialized.


100%|█████████▉| 356/357 [1:58:15<00:32, 32.91s/it]

Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/tmp/images_98.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/grayScaleEdgeDetection/images_98.npy.

  grayScaleEdgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/grayScaleEdgeDetection/images_98.npy
Finished images_98
Edge detection initialized.
Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/tmp/images_99.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/edgeDetection/images_99.npy.

  edgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/edgeDetection/images_99.npy
Grayscale edge detection initialized.


100%|██████████| 357/357 [1:58:39<00:00, 19.94s/it]

Processing on image /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/tmp/images_99.png is complete
Result is saved at /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/grayScaleEdgeDetection/images_99.npy.

  grayScaleEdgeDetection done: /content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/test/cenn_images/grayScaleEdgeDetection/images_99.npy
Finished images_99


In [ ]:
prepare_cenn_images(
    root_dir="/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/data/val",
    cnn=cnn,
)# Chay chua ruif as

## Step 2: Merge

In [ ]:
import os
import numpy as np
from tqdm import tqdm

def merge_image_variants(base_path="data/train"):
    # Input folders
    img_dir = os.path.join(base_path, "images")
    edge_dir = os.path.join(base_path, "cenn_images", "edgeDetection")
    gray_dir = os.path.join(base_path, "cenn_images", "grayScaleEdgeDetection")

    # Output folder
    out_dir = os.path.join(base_path, "merge_images")
    os.makedirs(out_dir, exist_ok=True)

    # Get all image filenames from the main image folder
    filenames = sorted([f for f in os.listdir(img_dir) if f.endswith(".npy")])

    for fname in tqdm(filenames):
        # print(f"Processing {fname} ...")

        # Build full paths for the three variants
        img_path = os.path.join(img_dir, fname)
        edge_path = os.path.join(edge_dir, fname)
        gray_path = os.path.join(gray_dir, fname)

        # Ensure all files exist
        if not (os.path.exists(edge_path) and os.path.exists(gray_path)):
            print(f"⚠️ Missing variants for {fname}, skipping...")
            continue

        # Load arrays
        img = np.load(img_path)
        edge = np.load(edge_path)
        gray = np.load(gray_path)

        # Stack along channel dimension
        merged = np.stack([img, edge, gray], axis=0)

        # Save output
        out_path = os.path.join(out_dir, fname)
        np.save(out_path, merged)

    print("✅ Merging completed.")


In [ ]:
import os
os.chdir('/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16')

In [ ]:
merge_image_variants("./data/train")

100%|██████████| 1095/1095 [10:15<00:00,  1.78it/s]

✅ Merging completed.


In [ ]:
merge_image_variants("./data/test")

100%|██████████| 357/357 [03:54<00:00,  1.52it/s]

✅ Merging completed.


In [ ]:
merge_image_variants("./data/val")

100%|██████████| 351/351 [03:23<00:00,  1.73it/s]

✅ Merging completed.
